# Chen SBML Model NNSE (Filler)
This notebook adapts the NNSE / NNSEFunc approach used for the Tyson model to the Chen (2004) budding yeast cell cycle model.
Uses the SBML model via Tellurium/RoadRunner and applies Gaussian mutations in normalized parameter space.

In [59]:
# Imports
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import platform
import roadrunner
import math
import multiprocessing as mp
from multiprocessing import Pool, cpu_count
import psutil
import warnings
import copy
import time
warnings.filterwarnings('ignore')

# Silence RoadRunner log messages
roadrunner.Logger.setLevel(roadrunner.Logger.LOG_CRITICAL)

print("🧪 ChenNNSEFunc_Filler: NNSE for Chen 2004 Yeast Cell Cycle Model")
print("="*60)

🧪 ChenNNSEFunc_Filler: NNSE for Chen 2004 Yeast Cell Cycle Model


In [60]:
# === CPU OPTIMIZATION DETECTION ===
def detect_cpu_config():
    """Detect optimal CPU configuration for Radeon/AMD processors"""
    cpu_count_total = cpu_count()
    cpu_info = platform.processor()
    
    # Check if this is an AMD/Radeon CPU
    is_amd = any(keyword in cpu_info.lower() for keyword in ['amd', 'radeon', 'ryzen', 'epyc', 'athlon'])
    
    if is_amd:
        try:
            physical_cores = psutil.cpu_count(logical=False)
            logical_cores = psutil.cpu_count(logical=True)
            optimal_workers = max(1, int(physical_cores * 0.8)) if physical_cores else max(1, int(cpu_count_total * 0.7))
            print(f"🔧 AMD/Radeon CPU detected: {cpu_info}")
            print(f"   Physical cores: {physical_cores}, Logical cores: {logical_cores}")
            print(f"   Optimal workers for simulation: {optimal_workers}")
            return optimal_workers, True
        except:
            optimal_workers = max(1, int(cpu_count_total * 0.7))
            print(f"🔧 AMD CPU detected (fallback): using {optimal_workers} workers")
            return optimal_workers, True
    else:
        optimal_workers = max(1, cpu_count_total - 1)
        print(f"🔧 Non-AMD CPU: using {optimal_workers} workers")
        return optimal_workers, False

OPTIMAL_WORKERS, IS_AMD_CPU = detect_cpu_config()

# Cross-platform path detection
def get_model_path():
    """Get the correct path to the SBML model file based on OS and available paths."""
    linux_path = "/home/gijs/Documents/OxfordEvolution/Yeast/Chen/chen_model.xml"
    Beecroft_path = "/home/b/bartholomeus/Documents/OxfordEvolution/Yeast/Chen/chen_model.xml"   
    mac_path = "/Users/gijsbartholomeus/Documents/STUDIE/OxfordEvolution/code/Yeast/Chen/chen_model.xml"
    
    if os.path.exists(linux_path):
        print("linux_path")
        return linux_path
    elif os.path.exists(mac_path):
        return mac_path
    elif os.path.exists(Beecroft_path):
        print("Beecroft_path")
        return Beecroft_path
    else:
        possible_relative_paths = ["chen_model.xml", "Chen/chen_model.xml", "../Chen/chen_model.xml", "../../Yeast/Chen/chen_model.xml"]
        for rel_path in possible_relative_paths:
            if os.path.exists(rel_path):
                return os.path.abspath(rel_path)
        raise FileNotFoundError(f"Could not find chen_model.xml in any of the expected locations:\n  Linux: {linux_path}\n  Mac: {mac_path}\nCurrent working directory: {os.getcwd()}\nPlatform: {platform.system()}")

# Load Chen 2004 budding yeast cell cycle model
model_path = get_model_path()
print(f"Loading model from: {model_path}")
rr = te.loadSBMLModel(model_path)
print("✓ Chen model loaded successfully")

🔧 Non-AMD CPU: using 11 workers
Beecroft_path
Loading model from: /home/b/bartholomeus/Documents/OxfordEvolution/Yeast/Chen/chen_model.xml
✓ Chen model loaded successfully


In [61]:
# === CONFIGURATION VARIABLES ===
# NNSE settings
N_STEPS = 2500           # Number of mutation steps to run
SIGMA = 0.01             # Gaussian mutation sigma in normalized u-space
N_Vec = 50               # Number of bins / maintained parameter vectors
MAX_VALUE = 250.0         # Maximum log value for bin thresholds (used as rough scale)
K_INITIAL = 1  # Number of worst positions to fill initially

# Simulation settings
T_START = 0.0
T_END = 500.0            # Chen system simulation time (minutes)
N_TIME_POINTS = 501
DIVERGENCE_THRESHOLD = 25000  # Reject divergent solutions

# Bin thresholds in objective space (these are used to accept/reject & permute)
bin_thresholds = np.linspace(0.0, MAX_VALUE, N_Vec + 1) + MAX_VALUE / max(1, N_Vec)

print(f"✓ Configuration:")
print(f"   N_STEPS: {N_STEPS}")
print(f"   N_Vec: {N_Vec}")
print(f"   SIGMA: {SIGMA}")
print(f"   T_END: {T_END} min")
print(f"   K_INITIAL: {K_INITIAL}")

✓ Configuration:
   N_STEPS: 2500
   N_Vec: 50
   SIGMA: 0.01
   T_END: 500.0 min
   K_INITIAL: 1


In [62]:
# === CORE FUNCTIONS ===

def get_kinetic_parameters(rr):
    """Get list of kinetic parameters, excluding regulatory switches/flags."""
    kinetic_params = []
    excluded_params = []
    
    for pid in rr.getGlobalParameterIds():
        value = rr.getValue(pid)
        param_lower = pid.lower()
        
        # Exclude non-kinetic parameters (switches, flags, totals)
        if (param_lower.endswith('t') and value in [0.0, 1.0]) or \
           (param_lower.startswith('d') and param_lower.endswith('n')) or \
           ('flag' in param_lower) or \
           ('switch' in param_lower) or \
           (value == 0.0) or \
           (pid in ['cell']) or \
           ('total' in param_lower and value in [0.0, 1.0]):
            excluded_params.append(pid)
        else:
            kinetic_params.append(pid)
    
    return kinetic_params, excluded_params

# Get kinetic parameters and cache default values
kinetic_params_all, excluded_params = get_kinetic_parameters(rr)
default_values_all = {pid: rr.getValue(pid) for pid in kinetic_params_all}

print(f"✓ Initial filtering: {len(kinetic_params_all)} kinetic parameters")
print(f"   Excluded {len(excluded_params)} non-kinetic parameters (switches/flags/totals)")

# === SECOND FILTERING STEP: Remove parameters with zero default values ===
# These cannot be normalized in the hypercube embedding (u = p / (2*p0))
kinetic_params = []
zero_params = []
for pid in kinetic_params_all:
    value = default_values_all[pid]
    if value == 0.0:
        zero_params.append(pid)
    else:
        kinetic_params.append(pid)

# Create dictionaries for mutable parameters only
default_values = {pid: default_values_all[pid] for pid in kinetic_params}
n_params = len(kinetic_params)

print(f"\n✓ Second filtering: Removed {len(zero_params)} parameters with zero default values")
if len(zero_params) > 0:
    print(f"   Zero-valued parameters (excluded from mutation): {zero_params}")

print(f"\n✓ FINAL: {n_params} parameters will be mutated")
print(f"   Total excluded: {len(excluded_params) + len(zero_params)}")

# Create p0_vec (default parameter values as vector) - only for mutable parameters
p0_vec = np.array([default_values[pid] for pid in kinetic_params], dtype=float)

# Verify no zeros in p0_vec
if np.any(p0_vec == 0.0):
    raise ValueError(f"ERROR: p0_vec still contains zeros! This will cause division by zero.")

# Print all mutable parameters with their default values
print(f"\n📋 ALL MUTABLE PARAMETERS WITH DEFAULT VALUES:")
print(f"{'Parameter':<20} {'Default Value':<20}")
print("-" * 40)
for pid in kinetic_params:
    print(f"{pid:<20} {default_values[pid]:<20.6e}")
print(f"\nTotal mutable parameters: {n_params}")

✓ Initial filtering: 156 kinetic parameters
   Excluded 7 non-kinetic parameters (switches/flags/totals)

✓ Second filtering: Removed 0 parameters with zero default values

✓ FINAL: 156 parameters will be mutated
   Total excluded: 7

📋 ALL MUTABLE PARAMETERS WITH DEFAULT VALUES:
Parameter            Default Value       
----------------------------------------
b0                   5.400000e-02        
bub2h                1.000000e+00        
bub2l                2.000000e-01        
C0                   4.000000e-01        
Dn3                  1.000000e+00        
ebudb5               1.000000e+00        
ebudn2               2.500000e-01        
ebudn3               5.000000e-02        
ec1b2                4.500000e-01        
ec1b5                1.000000e-01        
ec1k2                3.000000e-02        
ec1n2                6.000000e-02        
ec1n3                3.000000e-01        
ef6b2                5.500000e-01        
ef6b5                1.000000e-01        
ef6k2 

In [63]:
# === SIMULATION FUNCTIONS ===

def simulate_chen(rr, T=None, npoints=None):
    """Simulate Chen system and return CLB2 time series"""
    if T is None:
        T = T_END
    if npoints is None:
        npoints = N_TIME_POINTS
    
    # Set selections to minimum required
    rr.selections = ["time", "CLB2"]
    
    try:
        result = rr.simulate(0, T, npoints)
    except RuntimeError as e:
        return None, None
    
    time = result[:, 0]
    clb2 = result[:, 1]
    
    # Check for divergence
    if np.any(np.abs(clb2) > DIVERGENCE_THRESHOLD):
        return "divergent", None
    
    return time, clb2

def reset_model_to_params(rr, P_vec, kinetic_params_list, default_vals):
    """Reset model to default values and set new parameter vector"""
    # Reset to defaults
    for pid, default_val in default_vals.items():
        try:
            rr.setValue(pid, default_val)
        except RuntimeError:
            continue
    rr.resetAll()
    
    # Set new parameter values
    for i, pid in enumerate(kinetic_params_list):
        try:
            rr.setValue(pid, float(P_vec[i]))
        except (RuntimeError, IndexError):
            continue

print("✓ Simulation functions defined")

✓ Simulation functions defined


In [64]:
# === REFERENCE SIMULATION (p0) ===
print("Running reference simulation with default parameters...")
reset_model_to_params(rr, p0_vec, kinetic_params, default_values)
t0_ref, clb2_0_ref = simulate_chen(rr, T_END, N_TIME_POINTS)

if t0_ref is None or isinstance(t0_ref, str):
    raise RuntimeError("Reference simulation failed!")

print(f"✓ Reference simulation complete")
print(f"   Time range: {t0_ref[0]:.1f} - {t0_ref[-1]:.1f} min")
print(f"   CLB2 range: {np.min(clb2_0_ref):.3f} - {np.max(clb2_0_ref):.3f}")

Running reference simulation with default parameters...
✓ Reference simulation complete
   Time range: 0.0 - 500.0 min
   CLB2 range: 0.000 - 1.432


In [65]:
# === SIM FUNCTION (objective) ===
def sim(P_vec):
    """Simulate with parameter vector P_vec and compute integrated squared difference
    against the reference simulation (p0). Returns scalar squared-difference.
    """
    # Create a fresh model instance for this evaluation
    rr_local = te.loadSBMLModel(model_path)
    
    # Set parameters
    reset_model_to_params(rr_local, P_vec, kinetic_params, default_values)
    
    try:
        t, clb2 = simulate_chen(rr_local, T_END, N_TIME_POINTS)
        
        if t is None or isinstance(t, str):
            return float(np.inf)
        
        # Interpolate reference to match t
        clb2_0_interp = np.interp(t, t0_ref, clb2_0_ref)
        
        # Squared differences
        d_clb2_sq = (clb2 - clb2_0_interp) ** 2
        
        # Integrate over time
        integral = np.trapz(d_clb2_sq, t)
        return float(integral)
        
    except Exception as e:
        print(f'Warning: Simulation failed during sim(): {e}')
        return float(np.inf)

print('✓ sim() defined')

✓ sim() defined


In [66]:
# === ChenNNSEFunc: mutation + permutation + vacancy filling ===
def ChenNNSEFunc(X_list, fX_list):
    """Mutate each parameter vector, accept/reject based on thresholds, then permute.
    Args:
        X_list: list of parameter vectors (numpy arrays) or None for empty positions
        fX_list: list of scalar function values or None for empty positions
    Returns: (v_list, fv_list, fX_prime_list, swaps)
    """
    n = len(X_list)
    X_prime_list = []
    fX_prime_list = []
    # Mutation step
    for i in range(n):
        xi = X_list[i]
        fxi = fX_list[i]
        if xi is None or fxi is None:
            X_prime_list.append(None)
            fX_prime_list.append(None)
            continue
        # Normalize to u in [0,1] relative to p0_vec: u = xi / (2*p0_vec)
        u_vec = xi / (2.0 * p0_vec)
        # Gaussian mutation in u-space
        u_mut = (u_vec + np.random.normal(0.0, SIGMA, size=n_params)) % 1.0
        xi_prime = 2.0 * p0_vec * u_mut
        fxi_prime = sim(xi_prime)
        # Accept if fxi_prime <= threshold for this position; otherwise keep original
        if fxi_prime > bin_thresholds[i]:
            X_prime_list.append(xi.copy())
            fX_prime_list.append(fxi)
        else:
            X_prime_list.append(xi_prime)
            fX_prime_list.append(fxi_prime)
    # Permutation step (bubble up better entries)
    v_list = X_prime_list.copy()
    fv_list = fX_prime_list.copy()
    empty_before = set(i for i in range(n) if v_list[i] is None or fv_list[i] is None)
    swaps = []
    for i in range(n-1, 0, -1):
        if fv_list[i] is None and fv_list[i-1] is None:
            continue
        if fv_list[i] is not None and fv_list[i] <= bin_thresholds[i-1]:
            # swap i with i-1
            if v_list[i] is not None and v_list[i-1] is not None:
                v_list[i], v_list[i-1] = v_list[i-1].copy(), v_list[i].copy()
            else:
                v_list[i], v_list[i-1] = v_list[i-1], v_list[i]
            fv_list[i], fv_list[i-1] = fv_list[i-1], fv_list[i]
            swaps.append((i, i-1))
    # Fill newly empty positions by drawing random points until placed or attempts exhausted
    empty_after = set(i for i in range(n) if v_list[i] is None or fv_list[i] is None)
    newly_empty = empty_after - empty_before
    for empty_pos in newly_empty:
        placed = False
        for attempt in range(1000):
            u_rand = np.random.uniform(0.0, 1.0, size=n_params)
            x_new = 2.0 * p0_vec * u_rand
            fx_new = sim(x_new)
            # Place in the best position that is empty and where fx_new <= threshold
            for pos in range(n):
                if fx_new <= bin_thresholds[pos]:
                    if v_list[pos] is None or fv_list[pos] is None:
                        v_list[pos] = x_new
                        fv_list[pos] = fx_new
                        placed = True
                        break
            if placed:
                break
        if not placed:
            print(f'Warning: failed to fill empty position {empty_pos} after attempts')
    return v_list, fv_list, fX_prime_list, swaps

print('✓ ChenNNSEFunc defined')

✓ ChenNNSEFunc defined


In [ ]:
# === NNSE LOOP ===

# Helper function to format f values and y thresholds, handling None
def format_f_vals(fX_list, n):
    """Format f values and y thresholds for printing, handling None values.
    Returns two strings: (f_vals_str, y_vals_str)
    Always shows all f(xi) and y_i values.
    """
    indices = list(range(n))
    f_vals = [fX_list[i] if i < len(fX_list) else None for i in indices]
    f_vals_str = " ".join([f"{fx:<12.3e}" if fx is not None else f"{'---':<12}" for fx in f_vals])
    y_vals_str = " ".join([f"{bin_thresholds[i]:<12.3e}" for i in indices])
    return f_vals_str, y_vals_str

n = N_Vec
print(f'Starting Chen NNSE with n={n} vectors for {N_STEPS} steps...')

# Initialize population: keep best slots empty initially, fill worst K slots with random points
K = K_INITIAL
X_list = [None] * n
fX_list = [None] * n
for idx in range(n-K, n):
    u_rand = np.random.uniform(0.0, 1.0, size=n_params)
    xi = 2.0 * p0_vec * u_rand
    fxi = sim(xi)
    X_list[idx] = xi
    fX_list[idx] = fxi
filled = sum(1 for x in X_list if x is not None)

print(f"\n✓ Initialization complete ({filled} positions filled, {n-filled} empty)")

# Storage history
all_X = [[x.copy() if x is not None else None for x in X_list]]
all_fX = [[fx if fx is not None else None for fx in fX_list]]
all_swaps = [[]]

# Progress tracking variables
BURN_IN = int(0.5*N_STEPS)  # Wait for population to stabilize
swap_count = np.zeros(N_Vec + 1)  # Track swaps at each position
total_opportunities = np.zeros(N_Vec + 1)  # Track total chances to swap
is_burned_in = False
volume_ratio_history = []  # List of (step, volume_ratios) tuples
history_interval = 100  # Save every 100 steps after burn-in

# Progress loop
print_interval = max(1, N_STEPS // 50)

# Create table header
header_cols = [f"f(x{i})" for i in range(n)]
header = f"{'Step':<6} {'#swap':<6} {'#empty':<7} " + " ".join([f"{col:<12}" for col in header_cols]) + f" {'time':<12}"
print(f"\n{header}")

# Print y thresholds row header
y_header_cols = [f"y{i}" for i in range(n)]
y_header = f"{'':<6} {'':<6} {'':<7} " + " ".join([f"{col:<12}" for col in y_header_cols]) + f" {'':<12}"
print(y_header)

# Print y thresholds row once at the top (in italics)
_, y_vals_str = format_f_vals(fX_list, n)
print(f"\033[3m{'':<6} {'':<6} {'':<7} {y_vals_str} {'':<12}\033[0m")

# Print initial state
f_vals_str, y_vals_str = format_f_vals(fX_list, n)
empty_count = sum(1 for i in range(len(X_list)) if X_list[i] is None or fX_list[i] is None)
print(f"{'Init':<6} {'0':<6} {empty_count:<7} {f_vals_str} {'0.00 sec':<12}")

last_print_time = time.time()

for step in range(N_STEPS):
    # Store unmutated state (before mutation)
    fX_unmutated = [fx if fx is not None else None for fx in fX_list]
    
    # Apply ChenNNSEFunc
    X_list, fX_list, fX_mutated, swaps = ChenNNSEFunc(X_list, fX_list)
    
    # Track swap statistics (only after burn-in)
    if step >= BURN_IN:
        if not is_burned_in:
            # Reset counters at burn-in point
            swap_count[:] = 0
            total_opportunities[:] = 0
            is_burned_in = True
            print(f"\n=== BURN-IN COMPLETE at step {step} - Starting volume estimation ===\n")
            # Reprint table header after burn-in message
            print(f"{header}")
            print(y_header)
            print(f"\033[3m{'':<6} {'':<6} {'':<7} {y_vals_str} {'':<12}\033[0m")
        
        # Count swaps and opportunities at each position
        swap_set = set(swaps)  # swaps is a list of (i, i-1) tuples
        
        for i in range(1, len(fX_list)):  # positions 1 to n
            # Only count if both positions are filled (not None)
            if fX_list[i] is not None and fX_list[i-1] is not None:
                total_opportunities[i] += 1
                # Check if position i swapped with position i-1
                if (i, i-1) in swap_set:
                    swap_count[i] += 1
        
        # Store volume ratios at intervals for convergence tracking
        if (step - BURN_IN) % history_interval == 0 and step > BURN_IN:
            current_ratios = np.zeros(N_Vec + 1)
            for i in range(1, len(swap_count)):
                if total_opportunities[i] > 0:
                    current_ratios[i] = swap_count[i] / total_opportunities[i]
            volume_ratio_history.append((step, current_ratios.copy()))
    
    # Store trajectory
    all_X.append([x.copy() if x is not None else None for x in X_list])
    all_fX.append([fx if fx is not None else None for fx in fX_list])
    all_swaps.append(swaps)
    
    # Progress report in table format
    if (step + 1) % print_interval == 0 or step == 0:
        # Calculate time since last print
        current_time = time.time()
        interval_time = current_time - last_print_time
        last_print_time = current_time

        # Format time appropriately
        if interval_time > 60:
            interval_time_str = f"{interval_time/60:.2f} min"
        else:
            interval_time_str = f"{interval_time:.2f} sec"

        # Count empty positions in unmutated state
        empty_count_unmut = sum(1 for i in range(len(fX_unmutated)) if fX_unmutated[i] is None)

        # Print unmutated row
        f_vals_str, _ = format_f_vals(fX_unmutated, n)
        print(f"{step+1:<6} {'-':<6} {empty_count_unmut:<7} {f_vals_str} {interval_time_str:<12}")

        # Count empty positions in current state
        empty_count = sum(1 for i in range(len(fX_list)) if fX_list[i] is None)

        # Print mutated row
        f_vals_str, _ = format_f_vals(fX_mutated, n)
        print(f"{'':<6} {len(swaps):<6} {empty_count:<7} {f_vals_str} {interval_time_str:<12}")
        
        # If past burn-in, also print volume estimates
        if is_burned_in and step >= BURN_IN + 100:
            print(f"\n--- Volume Estimation (Step {step + 1}) ---")
            for i in range(1, min(6, len(swap_count))):
                if total_opportunities[i] > 0:
                    ratio = swap_count[i] / total_opportunities[i]
                    print(f"  Position {i}: swap_freq = {ratio:.4f} ({int(swap_count[i])}/{int(total_opportunities[i])})")

# After the loop, compute final volume estimates
if is_burned_in:
    print("\n=== Final Volume Ratios ===")
    volume_ratios = np.zeros(N_Vec + 1)
    for i in range(1, len(swap_count)):
        if total_opportunities[i] > 0:
            volume_ratios[i] = swap_count[i] / total_opportunities[i]
            print(f"V[{i-1}]/V[{i}] = {volume_ratios[i]:.6f}")

print('\n✓ Chen NNSE run complete')

# Simple visualization of best scalar objective over time
best_vals = []
for step_f in all_fX:
    best = np.nan
    for fx in step_f:
        if fx is not None and np.isfinite(fx):
            best = fx
            break
    best_vals.append(best)
plt.figure(figsize=(8,4))
plt.plot(best_vals, label='best f')
plt.yscale('log')
plt.xlabel('Step')
plt.ylabel('Best objective (log)')
plt.title('Chen NNSE: Best objective over steps')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Convert to numpy arrays for easier analysis
all_fX_clean = [[fx if fx is not None else np.nan for fx in step_fX] for step_fX in all_fX]
all_fX_array = np.array(all_fX_clean)  # Shape: (N_STEPS+1, n)

# For X (parameter vectors), replace None with NaN-filled arrays
all_X_clean = []
for step_X in all_X:
    step_X_clean = []
    for x in step_X:
        if x is not None:
            step_X_clean.append(x.copy())
        else:
            step_X_clean.append(np.full(n_params, np.nan))
    all_X_clean.append(step_X_clean)
all_X_array = np.array(all_X_clean)  # Shape: (N_STEPS+1, n, n_params)

Starting Chen NNSE with n=50 vectors for 2500 steps...

✓ Initialization complete (1 positions filled, 49 empty)

Step   #swap  #empty  f(x0)        f(x1)        f(x2)        f(x3)        f(x4)        f(x5)        f(x6)        f(x7)        f(x8)        f(x9)        f(x10)       f(x11)       f(x12)       f(x13)       f(x14)       f(x15)       f(x16)       f(x17)       f(x18)       f(x19)       f(x20)       f(x21)       f(x22)       f(x23)       f(x24)       f(x25)       f(x26)       f(x27)       f(x28)       f(x29)       f(x30)       f(x31)       f(x32)       f(x33)       f(x34)       f(x35)       f(x36)       f(x37)       f(x38)       f(x39)       f(x40)       f(x41)       f(x42)       f(x43)       f(x44)       f(x45)       f(x46)       f(x47)       f(x48)       f(x49)       time        
                      y0           y1           y2           y3           y4           y5           y6           y7           y8           y9           y10          y11          y12          y13       

In [ ]:
# ============================================================================
# === PLOT VOLUME RATIO EVOLUTION FOR POSITIONS 1-10 ===
# ============================================================================

if is_burned_in and len(volume_ratio_history) > 0:
    # Extract data for positions 1-10
    positions_to_plot = list(range(1, 11))  # Positions 1-10
    
    plt.figure(figsize=(12, 8))
    
    for pos in positions_to_plot:
        if pos < len(volume_ratio_history[0][1]):
            steps = [item[0] for item in volume_ratio_history]
            ratios = [item[1][pos] if item[1][pos] > 0 else np.nan for item in volume_ratio_history]
            plt.plot(steps, ratios, marker='o', label=f'Position {pos} (V[{pos-1}]/V[{pos}])', markersize=3)
    
    plt.xlabel('Step', fontsize=12)
    plt.ylabel('Volume Ratio V[i-1]/V[i]', fontsize=12)
    plt.title('Evolution of Volume Ratios for Positions 1-10', fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ============================================================================
# === COMPUTE AND PRINT VOLUME OF EACH SUBSET RELATIVE TO TOTAL VOLUME ===
# ============================================================================

if is_burned_in:
    print("\n=== Volume of Each Subset Relative to Total Volume ===")
    print("V[i]/V_total = Volume of {x: f(x) <= y_i} / Total Parameter Space Volume")
    print()
    
    # Find the maximum position with valid volume ratio data
    max_valid_pos = 0
    for i in range(1, len(volume_ratios)):
        if volume_ratios[i] > 0:
            max_valid_pos = i
    
    print(f"Using position {max_valid_pos} as maximum (V_total)")
    print()
    
    # Compute V[i]/V_total iteratively
    volume_fractions = np.full(N_Vec + 1, np.nan)
    
    # Start from the worst position (max_valid_pos): V[max_valid_pos]/V_total = 1.0
    volume_fractions[max_valid_pos] = 1.0
    
    # Work backwards: compute V[i]/V_total for i from max_valid_pos-1 down to 0
    for i in range(max_valid_pos - 1, -1, -1):
        # V[i]/V_total = product of ratios from i+1 to max_valid_pos
        product = 1.0
        for j in range(i + 1, max_valid_pos + 1):
            if j < len(volume_ratios) and volume_ratios[j] > 0:
                product *= volume_ratios[j]
            else:
                product = np.nan
                break
        volume_fractions[i] = product
    
    # Print results
    print(f"{'Position':<10} {'Threshold y_i':<15} {'V[i]/V_total':<15} {'Percentage':<12}")
    print("-" * 60)
    for i in range(max_valid_pos + 1):
        if not np.isnan(volume_fractions[i]):
            percentage = volume_fractions[i] * 100
            print(f"{i:<10} {bin_thresholds[i]:<15.6e} {volume_fractions[i]:<15.6e} {percentage:<12.4f}%")
        else:
            print(f"{i:<10} {bin_thresholds[i]:<15.6e} {'N/A':<15} {'N/A':<12}")
    
    # Plot volume fractions
    plt.figure(figsize=(10, 6))
    valid_indices = [i for i in range(max_valid_pos + 1) if not np.isnan(volume_fractions[i])]
    valid_fractions = [volume_fractions[i] for i in valid_indices]
    plt.plot(valid_indices, valid_fractions, marker='o', linewidth=2, markersize=6)
    plt.xlabel('Position i (threshold y_i)', fontsize=12)
    plt.ylabel('Volume Fraction V[i]/V_total', fontsize=12)
    plt.title(f'Volume of Each Subset Relative to Total Parameter Space (V_total = V[{max_valid_pos}])', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Cumulative volume plot: side by side log and linear
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    valid_indices = [i for i in range(max_valid_pos + 1) if not np.isnan(volume_fractions[i])]
    valid_fractions = [volume_fractions[i] for i in valid_indices]
    
    # Left plot: Log scale
    ax1.semilogy(valid_indices, valid_fractions, marker='o', linewidth=2, markersize=6, label='V[i]/V_total')
    ax1.set_xlabel('Position i (from best to worst, threshold y_i)', fontsize=12)
    ax1.set_ylabel('Cumulative Volume Fraction V[i]/V_total (log scale)', fontsize=12)
    ax1.set_title(f'Cumulative Volume (Log Scale)\nV_total = V[{max_valid_pos}], positions 0 to {max_valid_pos}', fontsize=13)
    ax1.grid(True, alpha=0.3, which='both')
    ax1.legend(fontsize=11)
    
    # Right plot: Linear scale
    ax2.plot(valid_indices, valid_fractions, marker='o', linewidth=2, markersize=6, label='V[i]/V_total', color='orange')
    ax2.set_xlabel('Position i (from best to worst, threshold y_i)', fontsize=12)
    ax2.set_ylabel('Cumulative Volume Fraction V[i]/V_total (linear scale)', fontsize=12)
    ax2.set_title(f'Cumulative Volume (Linear Scale)\nV_total = V[{max_valid_pos}], positions 0 to {max_valid_pos}', fontsize=13)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=11)
    
    # Add threshold annotations if not too many points
    if len(valid_indices) <= 20:
        step = max(1, len(valid_indices)//10)
        for idx in valid_indices[::step]:
            frac_idx = valid_indices.index(idx)
            # Log plot annotations
            ax1.annotate(f'y={bin_thresholds[idx]:.1f}', 
                        xy=(idx, valid_fractions[frac_idx]), 
                        xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.7)
            # Linear plot annotations
            ax2.annotate(f'y={bin_thresholds[idx]:.1f}', 
                        xy=(idx, valid_fractions[frac_idx]), 
                        xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.7)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================================
# === VISUALIZATION: PCA TRAJECTORY AND DISTRIBUTIONS ===
# ============================================================================

from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde

print("Preparing data for visualization...")

# Extract the best vector at each step
# Since vectors are sorted by f value, find the first non-NaN value (best available)
trajectory = []
squared_diffs = []
step_indices = []  # Keep track of which steps had valid data

for step_idx in range(len(all_X_array)):
    # Find the first non-NaN (best) vector at this step
    for pos_idx in range(n):
        if not np.isnan(all_fX_array[step_idx, pos_idx]):
            trajectory.append(all_X_array[step_idx, pos_idx])
            squared_diffs.append(all_fX_array[step_idx, pos_idx])
            step_indices.append(step_idx)
            break  # Found the best available vector, move to next step

trajectory = np.array(trajectory)
squared_diffs = np.array(squared_diffs)
step_indices = np.array(step_indices)

print(f"  Total steps: {len(all_X_array)}")
print(f"  Steps with valid data: {len(trajectory)}")
print(f"  First valid step: {step_indices[0] if len(step_indices) > 0 else 'None'}")
print(f"  Best f value trajectory: {len(squared_diffs)} points")

# ============================================================================
# === PCA TRAJECTORY PLOT ===
# ============================================================================

print("\nComputing PCA for trajectory visualization...")

# Normalize trajectory for PCA (use standardized parameters)
trajectory_normalized = (trajectory - trajectory.mean(axis=0)) / (trajectory.std(axis=0) + 1e-10)

# Compute PCA
pca = PCA(n_components=min(3, n_params))
trajectory_pca = pca.fit_transform(trajectory_normalized)

print(f"✓ PCA complete")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"  Total explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

# Simulate with first and last parameters for CLB2 comparison
print("Simulating with first and last parameters...")
P_first = trajectory[0]  # Best parameter vector at step 0
P_last = trajectory[-1]  # Best parameter vector at final step

# Simulate with first parameters
rr_first = te.loadSBMLModel(model_path)
reset_model_to_params(rr_first, P_first, kinetic_params, default_values)
t_first, clb2_first = simulate_chen(rr_first, T_END, N_TIME_POINTS)

# Simulate with last parameters
rr_last = te.loadSBMLModel(model_path)
reset_model_to_params(rr_last, P_last, kinetic_params, default_values)
t_last, clb2_last = simulate_chen(rr_last, T_END, N_TIME_POINTS)

print("✓ Simulations complete")

# Create figure with 3 subplots
fig = plt.figure(figsize=(20, 6))
gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

# Plot 2D PCA trajectory
ax1 = fig.add_subplot(gs[0, 0])
scatter = ax1.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], 
                     c=range(len(trajectory_pca)), cmap='viridis', 
                     s=20, alpha=0.6, edgecolors='none')
ax1.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], 'k-', alpha=0.3, linewidth=0.5)
ax1.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], 
           color='red', s=100, marker='o', label='Start', zorder=5)
ax1.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], 
           color='blue', s=100, marker='s', label='End', zorder=5)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
ax1.set_title('Chen NNSE Trajectory in PCA Space (PC1 vs PC2)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax1, label='Step')

# Second subplot: 3D plot or squared difference over time
if trajectory_pca.shape[1] >= 3:
    from mpl_toolkits.mplot3d import Axes3D
    ax2 = fig.add_subplot(gs[0, 1], projection='3d')
    scatter2 = ax2.scatter(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2],
                         c=range(len(trajectory_pca)), cmap='viridis', s=20, alpha=0.6)
    ax2.plot(trajectory_pca[:, 0], trajectory_pca[:, 1], trajectory_pca[:, 2], 
            'k-', alpha=0.3, linewidth=0.5)
    ax2.scatter(trajectory_pca[0, 0], trajectory_pca[0, 1], trajectory_pca[0, 2],
               color='red', s=100, marker='o', label='Start')
    ax2.scatter(trajectory_pca[-1, 0], trajectory_pca[-1, 1], trajectory_pca[-1, 2],
               color='blue', s=100, marker='s', label='End')
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})', fontsize=10)
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})', fontsize=10)
    ax2.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.2%})', fontsize=10)
    ax2.set_title('Chen NNSE Trajectory in PCA Space (3D)', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    plt.colorbar(scatter2, ax=ax2, label='Step')
else:
    # If only 2 components, show squared difference over time
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(range(len(squared_diffs)), squared_diffs, 'b-', linewidth=1, alpha=0.7)
    ax2.set_xlabel('Step', fontsize=12)
    ax2.set_ylabel('Squared Difference', fontsize=12)
    ax2.set_title('Best Squared Difference Over Time', fontsize=14, fontweight='bold')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)

# Third subplot: CLB2 comparison
ax3 = fig.add_subplot(gs[0, 2])
if t0_ref is not None and not isinstance(t0_ref, str):
    ax3.plot(t0_ref, clb2_0_ref, 'k-', lw=2, label='Reference (p0)', alpha=0.8)
if t_first is not None and not isinstance(t_first, str):
    ax3.plot(t_first, clb2_first, 'r-', lw=2, label='First parameter', alpha=0.7)
if t_last is not None and not isinstance(t_last, str):
    ax3.plot(t_last, clb2_last, 'b-', lw=2, label='Last parameter', alpha=0.7)
ax3.set_xlabel('Time (min)', fontsize=12)
ax3.set_ylabel('CLB2', fontsize=12)
ax3.set_title('CLB2 Comparison', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================================
# === DISTRIBUTION OF SQUARED DIFFERENCES ===
# ============================================================================

print("\nPlotting distribution of squared differences...")

# For distribution, use all squared differences from all vectors at all steps
squared_diffs_all = all_fX_array.flatten()  # All function values from all vectors

# Filter out infinite values
valid_mask = np.isfinite(squared_diffs_all)
squared_diffs_valid = squared_diffs_all[valid_mask]

print(f"  Valid values: {np.sum(valid_mask)}/{len(squared_diffs_all)}")
print(f"  Mean: {np.mean(squared_diffs_valid):.6e}")
print(f"  Median: {np.median(squared_diffs_valid):.6e}")
print(f"  Min: {np.min(squared_diffs_valid):.6e}")
print(f"  Max: {np.max(squared_diffs_valid):.6e}")

# Create distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram with linear bins
ax1 = axes[0]
n_bins_hist = 50
counts, bins_hist, patches = ax1.hist(squared_diffs_valid, bins=n_bins_hist, 
                                      edgecolor='black', alpha=0.7, color='steelblue',
                                      density=True)
ax1.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
           linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
ax1.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
           linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
ax1.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Squared Differences (Histogram)', 
             fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Kernel density estimate (KDE) with linear scale
ax2 = axes[1]
if len(squared_diffs_valid) > 1:
    kde = gaussian_kde(squared_diffs_valid)
    x_kde = np.linspace(squared_diffs_valid.min(), squared_diffs_valid.max(), 200)
    density = kde(x_kde)
    ax2.plot(x_kde, density, 'b-', linewidth=2, label='KDE')
    ax2.fill_between(x_kde, 0, density, alpha=0.3, color='steelblue')
    ax2.axvline(np.mean(squared_diffs_valid), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(squared_diffs_valid):.4e}')
    ax2.axvline(np.median(squared_diffs_valid), color='green', linestyle='--', 
               linewidth=2, label=f'Median: {np.median(squared_diffs_valid):.4e}')
    ax2.set_xlabel('Squared Difference', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Not enough data for KDE', 
            ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Distribution of Squared Differences (KDE)', 
                 fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")